# Lezione 10: Database NoSQL

**Obiettivi di apprendimento:**
- Spiegare perché esiste il NoSQL.
- Distinguere i quattro tipi (chiave-valore, documenti, colonne, grafi) e il loro caso d'uso tipico.
- Rappresentare un documento NoSQL come struttura Python/JSON.

**Durata stimata:** 110 min


## Perché serve qualcosa di più dell'SQL?

Nella lezione precedente abbiamo visto che i database relazionali sono straordinariamente affidabili grazie ad ACID. Ma hanno un punto debole: richiedono uno schema fisso, e non sempre scalano bene quando il volume di letture e scritture è enorme (pensa a un social media con centinaia di milioni di utenti che scrivono contemporaneamente).

**NoSQL** ("Not Only SQL") è la famiglia di database che rinunciano a parte di quella rigidità in cambio di flessibilità e scalabilità massiccia. La spinta storica venne da un whitepaper di Google del 2006 su un sistema chiamato BigTable, che ispirò alternative come Cassandra o HBase.

Non c'è "un" database NoSQL: ci sono **quattro tipi distinti**. Simuliamo ognuno con Python puro per capire l'idea di fondo di ciascuno.

In [ ]:
# 1. CHIAVE-VALORE (come Redis, Memcached, DynamoDB)
# La struttura più semplice possibile: ogni chiave punta a un valore,
# senza che il sistema sappia (né gli importi) cosa c'è dentro il valore.
sesiones_activas = {
    "usuario_284": "activo_desde_10:32",
    "usuario_591": "activo_desde_10:47",
    "usuario_112": "activo_desde_11:03",
}

print("Sessione dell'usuario_591:", sesiones_activas["usuario_591"])
print("\nCaso d'uso tipico: sessioni, cache, raccomandazioni rapide.")

## Documenti: quando ogni record ha la sua forma

In una tabella SQL, tutte le righe condividono esattamente le stesse colonne. In un database **basato su documenti** (MongoDB, CouchDB, Cloudant...), ogni "documento" può avere una struttura diversa — ideale per cataloghi eCommerce, dove una maglietta ha taglia e colore, e delle cuffie hanno autonomia e bluetooth.

In [ ]:
import json

productos = [
    {
        "id_producto": "P001",
        "nombre": "Camiseta básica",
        "categoria": "ropa",
        "atributos": {"talla": "M", "color": "azul"},
        "precio": 15.99,
    },
    {
        "id_producto": "P002",
        "nombre": "Auriculares inalámbricos",
        "categoria": "electrónica",
        "atributos": {"autonomia_horas": 20, "bluetooth": True},
        "precio": 39.90,
    },
]

# json.dumps mostra il documento come verrebbe salvato in Mongo
print(json.dumps(productos[0], indent=2, ensure_ascii=False))

def buscar_por_categoria(coleccion, categoria):
    return [doc for doc in coleccion if doc["categoria"] == categoria]

print("\nProdotti di elettronica:")
print(buscar_por_categoria(productos, "electrónica"))

## Colonnare: salvare per colonna invece che per riga

I database **colonnari** (Cassandra, HBase) raggruppano i valori di una stessa colonna insieme in memoria, invece di raggruppare per riga come fa SQL. Questo li rende straordinariamente veloci per scritture massicce e per leggere una sola colonna senza toccare le altre — il caso tipico sono le serie temporali di sensori IoT.

In [ ]:
# Invece di una lista di righe, una lista per colonna.
lecturas_sensores = {
    "id_sensor":    ["S1", "S2", "S3", "S4"],
    "temperatura":  [21.3, 19.8, 22.1, 20.5],
    "humedad":      [55, 60, 48, 58],
}

print("Tutte le temperature in un colpo solo, senza toccare altre colonne:")
print(lecturas_sensores["temperatura"])

## Grafi: quando ciò che conta è la relazione

I database a **grafi** (Neo4j, CosmosDB) rappresentano i dati come nodi collegati da relazioni. Sono il modello naturale per social media, rilevamento di frodi e sistemi di raccomandazione — domande del tipo "chi sono gli amici dei miei amici?".

In [ ]:
red_amistades = {
    "Ana":   ["Luis", "Carla"],
    "Luis":  ["Ana", "Diego"],
    "Carla": ["Ana", "Marta"],
    "Diego": ["Luis"],
    "Marta": ["Carla"],
}

# BFS (visita in ampiezza): prima gli amici diretti (livello 1),
# poi gli amici di quegli amici (livello 2), senza ripetere nessuno.
def amigos_de_amigos(grafo, persona):
    visitados = {persona}
    cola = [(vecino, 1) for vecino in grafo.get(persona, [])]
    resultado_nivel_2 = []
    while cola:
        actual, nivel = cola.pop(0)
        if actual in visitados:
            continue
        visitados.add(actual)
        if nivel == 2:
            resultado_nivel_2.append(actual)
            continue
        for vecino in grafo.get(actual, []):
            if vecino not in visitados:
                cola.append((vecino, nivel + 1))
    return resultado_nivel_2

print("Amici degli amici di Ana:", amigos_de_amigos(red_amistades, "Ana"))

## RDBMS vs. NoSQL: la tabella da avere chiara

| | RDBMS | NoSQL |
|---|---|---|
| Schema | Rigido | Flessibile |
| Garanzie | ACID | Privilegia la scalabilità |
| Maturità | Molto matura | Più recente |
| Hardware | Richiede di solito macchine potenti | Scala a basso costo con hardware di base |

La scelta reale dipende dal caso d'uso, dal tipo di dato, dalle prestazioni, dal volume e dagli standard dell'organizzazione. Non è raro combinare più soluzioni contemporaneamente all'interno della stessa azienda.

## Riepilogo della lezione

- Il NoSQL esiste perché gli RDBMS non si adattano bene a tutti i tipi di dati né a tutte le scale.
- Quattro tipi: **chiave-valore** (dict semplice), **documenti** (lista di dict annidati), **colonnare** (dict di liste), **grafi** (dict di adiacenza).
- Ogni tipo ha il suo caso d'uso ideale: cache/sessioni, eCommerce/CRM, serie temporali/IoT, social media/frodi.
- Possiamo simulare fedelmente i quattro tipi con strutture native di Python.

**Capitolo del manuale:** [12 — Database NoSQL](../manual/12_bases_nosql_IT.md)

**Notebook di pratica:** [clase10_practica_bases_nosql_IT.ipynb](../notebooks_practica/clase10_practica_bases_nosql_IT.ipynb)